# Appendix A2 — LangChain from Zero

**Who this is for:** a complete beginner. By the end you'll comfortably use LangChain's building
blocks — chat models, messages, prompt templates, **LCEL** (the pipe composition), output
parsers, tools, document splitting, embeddings + vector stores + retrievers, a full **RAG chain**,
conversation memory, and a first agent.

**How to read it:** each idea is explained, then a **runnable code cell** shows it. Run top to
bottom.

> **LangChain 1.x** (current major version). APIs here are verified against langchain 1.3 /
> langchain-core 1.6 — some names differ from older 0.x tutorials. **No API key required:** every
> cell runs with a deterministic *fake* model. To use a real LLM, add a Colab secret (shown below).

### The ecosystem in one paragraph
- **`langchain-core`** — the primitives: messages, prompts, the `Runnable` interface, parsers.
- **`langchain`** — higher-level pieces (agents, some chains) built on core.
- **integration packages** (`langchain-openai`, …) — adapters to specific model providers.
- **`langgraph`** (Appendix A3) — stateful, multi-step agent graphs.
- **`langsmith`** (Appendix A4) — tracing + evaluation.

In [1]:
# === Chapter A2 · standard bootstrap (identical pattern in every notebook) ===
# Runs standalone on a fresh Google Colab VM *or* a local checkout.
import os, sys, subprocess

REPO_URL = "https://github.com/rsalehin/patent-rag-masterclass"
NEED_OCR = False
IN_COLAB = "google.colab" in sys.modules


def _clone_repo(url, target):
    """Clone the repo on Colab. For a PRIVATE repo, authenticate with a GitHub token read from
    Colab Secrets (key 'GITHUB_TOKEN') or the GITHUB_TOKEN env var. The token is never printed."""
    token = None
    try:
        from google.colab import userdata  # type: ignore
        token = userdata.get("GITHUB_TOKEN")
    except Exception:
        token = os.environ.get("GITHUB_TOKEN")
    auth_url = url
    if token and url.startswith("https://github.com/"):
        auth_url = url.replace("https://github.com/", f"https://{token}@github.com/")
    r = subprocess.run(["git", "clone", "--depth", "1", auth_url, target],
                       stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)  # avoid leaking the token
    if r.returncode != 0:
        raise RuntimeError(
            "git clone failed. This is a PRIVATE repo, so Colab needs a GitHub token:\n"
            "  1) Create a token (scope: repo) at https://github.com/settings/tokens\n"
            "  2) In Colab, open the key icon (Secrets) in the left sidebar, add a secret named\n"
            "     GITHUB_TOKEN, paste the token, and enable 'Notebook access'.\n"
            "  3) Re-run this cell.\n"
            "  (Alternatively, make the GitHub repo public — then no token is needed.)")


if IN_COLAB:
    target = "/content/patent-rag-masterclass"
    if not os.path.isdir(target):
        if not REPO_URL:
            raise RuntimeError("Set REPO_URL to this repo's GitHub URL (see README.md).")
        _clone_repo(REPO_URL, target)
    os.chdir(target)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
    if NEED_OCR:
        subprocess.run(["apt-get", "install", "-y", "-q", "tesseract-ocr"], check=False)

# Ensure the repo root (containing patentrag/) is importable.
for _cand in [os.getcwd()] + [os.path.dirname(os.getcwd())]:
    if os.path.isdir(os.path.join(_cand, "patentrag")):
        if _cand not in sys.path:
            sys.path.insert(0, _cand)
        break

from patentrag import bootstrap as bs
bs.setup_environment(REPO_URL, need_ocr=NEED_OCR)
bs.set_seeds()
_env = bs.environment_report()
print("Chapter A2 bootstrap OK")
print("  Python", _env["python"], "| Colab:", _env["in_colab"], "| CPU cores:", _env["cpu_count"])
print("  torch", _env["torch"], "| CUDA:", _env["cuda_available"], "| tesseract:", _env["tesseract"])

Chapter A2 bootstrap OK
  Python 3.12.10 | Colab: False | CPU cores: 24
  torch 2.12.0.dev20260304+cu130 | CUDA: True | tesseract: True


In [2]:
# Install the LangChain / LangGraph / LangSmith stack (extra deps for the appendices).
# No-op locally if already installed; installs on a fresh Colab VM.
import sys, subprocess, os
if os.path.exists("requirements-appendix.txt"):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-appendix.txt"], check=True)
else:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "langchain==1.3.18", "langchain-core==1.6.1", "langchain-text-splitters==1.1.2",
                    "langgraph==1.2.11", "langsmith==0.11.2", "langchain-openai==1.6.0"], check=True)
print("LangChain stack ready.")

LangChain stack ready.


In [3]:
# --- optional real LLM + a deterministic offline fallback -------------------------------------
# Everything in these appendices runs with NO API key using deterministic fake models. To use a
# REAL model, add a Colab Secret (key icon, left sidebar) named LLM_API_KEY (any OpenAI-compatible
# endpoint; optionally LLM_BASE_URL and LLM_MODEL), or OPENAI_API_KEY. For LangSmith tracing add
# LANGSMITH_API_KEY. Secrets are pulled into environment variables here; nothing is printed.
import os
def _load_secret(name):
    try:
        from google.colab import userdata  # type: ignore
        v = userdata.get(name)
        if v:
            os.environ[name] = v
    except Exception:
        pass
for _n in ["OPENAI_API_KEY", "LLM_API_KEY", "LLM_BASE_URL", "LLM_MODEL", "LANGSMITH_API_KEY"]:
    _load_secret(_n)

LIVE_LLM = bool(os.environ.get("LLM_API_KEY") or os.environ.get("OPENAI_API_KEY"))

def get_chat_model(fake_responses=None, temperature: float = 0.0):
    """Return a real ChatOpenAI if a key is configured, else a deterministic fake chat model."""
    if LIVE_LLM:
        from langchain_openai import ChatOpenAI
        return ChatOpenAI(model=os.environ.get("LLM_MODEL", "gpt-4o-mini"),
                          base_url=os.environ.get("LLM_BASE_URL"),
                          api_key=os.environ.get("LLM_API_KEY") or os.environ.get("OPENAI_API_KEY"),
                          temperature=temperature)
    from langchain_core.language_models.fake_chat_models import GenericFakeChatModel
    return GenericFakeChatModel(messages=iter(fake_responses or ["(deterministic fake-model answer)"]))

# A scripted tool-calling model so the REAL agent APIs can run offline (it replays AIMessages,
# including tool_calls, and implements bind_tools so agents accept it).
from langchain_core.language_models.chat_models import BaseChatModel
from langchain_core.outputs import ChatResult, ChatGeneration
from pydantic import PrivateAttr
class ScriptedChatModel(BaseChatModel):
    responses: list
    _i: int = PrivateAttr(default=0)
    def _generate(self, messages, stop=None, run_manager=None, **kw):
        msg = self.responses[min(self._i, len(self.responses) - 1)]
        self._i += 1
        return ChatResult(generations=[ChatGeneration(message=msg)])
    def bind_tools(self, tools, **kw):
        return self
    @property
    def _llm_type(self):
        return "scripted"

print("LLM helpers ready. Live model configured:", LIVE_LLM)

LLM helpers ready. Live model configured: False


## 1. Chat models & messages

Modern LLMs are **chat** models: they take a list of **messages** and return an **AIMessage**.
The three everyday roles are **System** (instructions/persona), **Human** (the user), and **AI**
(the model). `get_chat_model()` returns a real model if you configured a key, otherwise a
deterministic fake so this notebook always runs.

In [4]:
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
llm = get_chat_model(["The capital of France is Paris."])
response = llm.invoke([
    SystemMessage(content="You are a concise geography tutor."),
    HumanMessage(content="What is the capital of France?"),
])
print(type(response).__name__, "->", response.content)

AIMessage -> The capital of France is Paris.


## 2. Three ways to call a model: `invoke`, `batch`, `stream`

Every LangChain component shares one interface (the **Runnable** interface):
- **`invoke(x)`** — one input → one output.
- **`batch([x1, x2])`** — many inputs in parallel.
- **`stream(x)`** — yields the answer in **chunks** as it's produced (for typing-effect UIs).

In [5]:
m1 = get_chat_model(["one", "two"])
print("batch :", [r.content for r in m1.batch(["q1", "q2"])])
m2 = get_chat_model(["streaming happens token by token"])
print("stream:", [chunk.content for chunk in m2.stream("go")][:6], "...")

batch : ['one', 'two']
stream: ['streaming', ' ', 'happens', ' ', 'token', ' '] ...


## 3. Prompt templates

Hard-coding prompts is brittle. A **`ChatPromptTemplate`** is a reusable prompt with
**`{placeholders}`** filled at call time. `MessagesPlaceholder` reserves a slot for a whole list
of messages (e.g. chat history).

In [6]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a {domain} expert. Answer in one sentence."),
    MessagesPlaceholder("history"),
    ("human", "{question}"),
])
filled = prompt.invoke({"domain": "patent", "history": [HumanMessage("hi")], "question": "What is prior art?"})
for m in filled.to_messages():
    print(f"  [{m.type}] {m.content}")

  [system] You are a patent expert. Answer in one sentence.
  [human] hi
  [human] What is prior art?


## 4. LCEL — compose with the pipe `|`

**LCEL** (LangChain Expression Language) chains Runnables with the **`|`** operator: the output
of one becomes the input of the next. The canonical chain is **`prompt | model | parser`**. The
resulting chain is *itself* a Runnable (so it also has `invoke`/`batch`/`stream`).

In [7]:
from langchain_core.output_parsers import StrOutputParser
tmpl = ChatPromptTemplate.from_template("Give the capital of {country}.")
model = get_chat_model(["Paris"])
chain = tmpl | model | StrOutputParser()      # a 3-step pipeline
print("chain result:", chain.invoke({"country": "France"}))
print("type of chain:", type(chain).__name__, "(itself a Runnable)")

chain result: Paris
type of chain: RunnableSequence (itself a Runnable)


## 5. Output parsers — turn model text into usable data

Models emit **text**; you usually want **structure**. Output parsers sit at the end of a chain:
- **`StrOutputParser`** — just the string.
- **`JsonOutputParser`** — parse a JSON object.
- **`PydanticOutputParser`** — parse into a validated **Pydantic** model (see Appendix A1), and it
  can inject **format instructions** into your prompt telling the model exactly what shape to emit.

In [8]:
from langchain_core.output_parsers import JsonOutputParser, PydanticOutputParser
from pydantic import BaseModel, Field

class PatentFact(BaseModel):
    number: str = Field(description="publication number")
    topic: str = Field(description="one-word topic")

parser = PydanticOutputParser(pydantic_object=PatentFact)
# a real model would follow parser.get_format_instructions(); the fake returns valid JSON directly
fake = get_chat_model(['{"number": "US9081550B2", "topic": "speech"}'])
result = (fake | parser).invoke("extract a patent fact")
print("parsed object:", result, "| type:", type(result).__name__)
print("\nformat instructions the parser would add to a real prompt:\n", parser.get_format_instructions()[:160], "...")

parsed object: number='US9081550B2' topic='speech' | type: PatentFact

format instructions the parser would add to a real prompt:
 The output should be formatted as a JSON instance that conforms to the JSON schema below.

As an example, for the schema {"properties": {"foo": {"title": "Foo", ...


### The modern shortcut: `.with_structured_output()`

With a **real** tool-calling model you can skip manual parsing: `model.with_structured_output(
PatentFact)` returns objects directly. (It relies on the provider's tool-calling, which the
offline fake doesn't implement — so above we used `PydanticOutputParser`, which works with *any*
text model.)

In [9]:
if LIVE_LLM:
    structured = get_chat_model().with_structured_output(PatentFact)
    print(structured.invoke("Give a patent fact about US9081550B2 (speech)."))
else:
    print("SKIPPED (no live model): .with_structured_output needs provider tool-calling.")
    print("Offline we used PydanticOutputParser above, which achieves the same result with a fake model.")

SKIPPED (no live model): .with_structured_output needs provider tool-calling.
Offline we used PydanticOutputParser above, which achieves the same result with a fake model.


## 6. Composing Runnables: parallel, passthrough, lambda

Beyond a straight pipe you can branch and transform:
- **`RunnableParallel`** (or a plain dict) — run several Runnables on the same input, collect a dict.
- **`RunnablePassthrough`** — pass the input through unchanged (useful inside a dict).
- **`RunnableLambda`** — wrap any Python function as a Runnable.

In [10]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableLambda
pipeline = RunnableParallel(
    original=RunnablePassthrough(),
    shout=RunnableLambda(lambda s: s.upper()),
    length=RunnableLambda(len),
)
print(pipeline.invoke("hello"))

{'original': 'hello', 'shout': 'HELLO', 'length': 5}


## 7. Tools — functions the model can call

A **tool** is a Python function the LLM can invoke to *act* (search, calculate, fetch). The
**`@tool`** decorator turns a function into a tool: its name, docstring, and typed arguments
become a schema the model understands. (Letting a model *choose* tools is agents — §11 / A3.)

In [11]:
from langchain_core.tools import tool

@tool
def word_count(text: str) -> int:
    """Return the number of words in the given text."""
    return len(text.split())

print("tool name       :", word_count.name)
print("tool description:", word_count.description)
print("tool args schema:", word_count.args)
print("direct call     :", word_count.invoke({"text": "one two three"}))

tool name       : word_count
tool description: Return the number of words in the given text.
tool args schema: {'text': {'title': 'Text', 'type': 'string'}}
direct call     : 3


## 8. Documents & text splitters

Retrieval works over **`Document`** objects (`page_content` + `metadata`). Long text must be
**split** into chunks first. `RecursiveCharacterTextSplitter` splits on natural boundaries
(paragraphs → sentences → words) up to a size, with optional overlap. (Appendix chapter 05 of the
main series covers *why* chunking matters for citability.)

In [12]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
long_text = ("Approximate nearest neighbor search finds close vectors quickly. "
             "HNSW builds a layered graph. Product quantization compresses vectors. ") * 6
splitter = RecursiveCharacterTextSplitter(chunk_size=120, chunk_overlap=20)
chunks = splitter.create_documents([long_text])
print(f"{len(chunks)} chunks; first chunk:\n  {chunks[0].page_content!r}")

C:\Users\rsalehin\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


8 chunks; first chunk:
  'Approximate nearest neighbor search finds close vectors quickly. HNSW builds a layered graph. Product quantization'


## 9. Embeddings + vector store + retriever

An **embedding model** maps text → a vector; a **vector store** indexes those vectors for
similarity search; a **retriever** is the Runnable you query. We use a *deterministic fake*
embedding + an in-memory store so this runs offline (swap in a real embedding model in production).

In [13]:
from langchain_core.embeddings import DeterministicFakeEmbedding
from langchain_core.vectorstores import InMemoryVectorStore
embeddings = DeterministicFakeEmbedding(size=128)
store = InMemoryVectorStore.from_documents([
    Document(page_content="HNSW is a graph index for approximate nearest neighbor search."),
    Document(page_content="Product quantization compresses vectors to save memory."),
    Document(page_content="A tomato soup recipe with basil and cream."),
], embeddings)
retriever = store.as_retriever(search_kwargs={"k": 2})
hits = retriever.invoke("how to search vectors quickly")
print("retrieved:", [d.page_content[:40] for d in hits])

retrieved: ['A tomato soup recipe with basil and crea', 'HNSW is a graph index for approximate ne']


## 10. A full RAG chain in LCEL

Retrieval-Augmented Generation = **retrieve context → stuff it into a prompt → generate**. Here
is the whole thing as one LCEL pipeline. (The fake model returns a canned answer; the point is the
*plumbing* — with a real model the answer is grounded in the retrieved context.)

In [14]:
def format_docs(docs):
    return "\n".join(f"- {d.page_content}" for d in docs)

rag_prompt = ChatPromptTemplate.from_template(
    "Answer the question using ONLY this context:\n{context}\n\nQuestion: {question}\nAnswer:")
rag_model = get_chat_model(["HNSW is a graph-based index for fast approximate nearest-neighbor search."])
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | rag_prompt
    | rag_model
    | StrOutputParser()
)
print(rag_chain.invoke("What is HNSW?"))

HNSW is a graph-based index for fast approximate nearest-neighbor search.


## 11. Conversation memory

By default a chain is **stateless** — it forgets previous turns. **`RunnableWithMessageHistory`**
wraps a chain and stores per-session history, replaying it into the `MessagesPlaceholder` each
call. (For robust, checkpointed memory, LangGraph in Appendix A3 is the modern approach.)

In [15]:
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.chat_history import InMemoryChatMessageHistory

chat_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant."),
    MessagesPlaceholder("history"),
    ("human", "{input}"),
])
convo_model = get_chat_model(["Nice to meet you, Ada!", "Your name is Ada."])
base_chain = chat_prompt | convo_model | StrOutputParser()

_sessions = {}
def get_history(session_id):
    return _sessions.setdefault(session_id, InMemoryChatMessageHistory())

with_memory = RunnableWithMessageHistory(base_chain, get_history,
                                         input_messages_key="input", history_messages_key="history")
cfg = {"configurable": {"session_id": "demo"}}
print("turn 1:", with_memory.invoke({"input": "Hi, I'm Ada."}, config=cfg))
print("turn 2:", with_memory.invoke({"input": "What's my name?"}, config=cfg))
print("stored messages:", len(get_history("demo").messages))

turn 1: Nice to meet you, Ada!
turn 2: Your name is Ada.
stored messages: 4


C:\Users\rsalehin\AppData\Roaming\Python\Python312\site-packages\IPython\core\interactiveshell.py:3701: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


## 12. A first agent (preview)

An **agent** lets the model *decide* which tools to call, in a loop, until it can answer.
LangChain 1.x provides **`create_agent`**. It needs a tool-calling model; to run offline we drive
it with a small **scripted** model (`ScriptedChatModel`) that replays a tool call then a final
answer. Appendix **A3 (LangGraph)** builds agents from scratch and explains the loop.

In [16]:
from langchain.agents import create_agent

@tool
def lookup_patent(topic: str) -> str:
    """Look up a patent by topic."""
    return "US9081550B2 — Adding speech capabilities to existing GUI applications."

driver = ScriptedChatModel(responses=[
    AIMessage(content="", tool_calls=[{"name": "lookup_patent", "args": {"topic": "voice UI"}, "id": "c1"}]),
    AIMessage(content="The patent about voice UIs is US9081550B2."),
])
agent = create_agent(driver, [lookup_patent])
result = agent.invoke({"messages": [HumanMessage("Which patent is about voice UIs?")]})
for m in result["messages"]:
    print(f"  [{m.type:9}] {m.content[:56]!r}")

  [human    ] 'Which patent is about voice UIs?'
  [ai       ] ''
  [tool     ] 'US9081550B2 — Adding speech capabilities to existing GUI'
  [ai       ] 'The patent about voice UIs is US9081550B2.'


## 13. Tie-in: RAG over the real patent corpus

The same RAG chain, now indexing real bundled patent text. (Fake embeddings + fake model keep it
offline and deterministic; swap in real ones and the answer becomes grounded.)

In [17]:
import json
docs = []
for p in sorted((bs.DATA / "corpus").glob("US*.json"))[:5]:
    d = json.loads(p.read_text(encoding="utf-8"))
    docs.append(Document(page_content=f"{d['title']}. {d['abstract'][:200]}",
                         metadata={"pub": d["publication_number"]}))
patent_store = InMemoryVectorStore.from_documents(docs, embeddings)
patent_ret = patent_store.as_retriever(search_kwargs={"k": 2})
found = patent_ret.invoke("retrieval aware embeddings for search")
print("retrieved real patents:", [d.metadata["pub"] for d in found])

retrieved real patents: ['US10261954B2', 'US11093561B2']


### You now know the core of LangChain

messages · chat models · invoke/batch/stream · prompt templates · **LCEL (`|`)** · output parsers
(Str/Json/Pydantic) · `with_structured_output` · parallel/passthrough/lambda · tools · documents &
splitters · embeddings + vector store + retriever · **RAG chains** · conversation memory · a first
agent. Next: **A3 — LangGraph** (stateful multi-step agents) and **A4 — LangSmith** (tracing +
evaluation).

## Chapter invariants

In [18]:
# (fake models have single-use response iterators, so build a fresh chain here)
_check = ChatPromptTemplate.from_template("{x}") | get_chat_model(["ok"]) | StrOutputParser()
assert _check.invoke({"x": "hi"}) == "ok"                  # LCEL pipe works
assert isinstance(result["messages"][-1].content, str) and result["messages"][-1].content  # agent answered
assert len(found) == 2 and all("pub" in d.metadata for d in found)   # real-corpus retrieval
assert isinstance(PatentFact(number="US1", topic="x"), PatentFact)   # Pydantic parse target
print("All Appendix A2 invariants hold.")

All Appendix A2 invariants hold.


In [19]:
# === Chapter A2 validation footer ===
import time, platform, sys, importlib.metadata as _md
_pkgs = ['langchain', 'langchain-core', 'langgraph', 'langsmith']
print("Chapter A2 — environment")
print("  Python :", sys.version.split()[0], "on", platform.system(), platform.release())
for _p in _pkgs:
    try: print(f"  {_p:24}: {_md.version(_p)}")
    except Exception: print(f"  {_p:24}: (not installed)")
print()
print("CHAPTER A2 VALIDATION: PASS")

Chapter A2 — environment
  Python : 3.12.10 on Windows 11
  langchain               : 1.3.18
  langchain-core          : 1.6.1
  langgraph               : 1.2.11
  langsmith               : 0.11.2

CHAPTER A2 VALIDATION: PASS
